# 02 — Gender Prediction from First Names (Baseline)

Predicts binary gender (M / F) from a first name using a **character-frequency feature matrix** (26 features).

This notebook explores and compares five classical ML models as a baseline. For the production model using character n-gram TF-IDF + LightGBM, see **notebook 05**.

**Models Compared:**
1. Logistic Regression
2. K-Nearest Neighbors
3. Naive Bayes (Multinomial)
4. Random Forest
5. Support Vector Machine

**Data:** `data/processed/Unique_Names_Till_2021.csv` — 101,338 deduplicated SSA names (1880–2021)

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
)

# ── Paths ──────────────────────────────────────────────────────────────────────
BASE        = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROCESSED   = os.path.join(BASE, 'data', 'processed')

RANDOM_SEED = 42
print('Base:', BASE)

## 1. Load Data

In [ ]:
df = pd.read_csv(os.path.join(PROCESSED, 'Unique_Names_Till_2021.csv'))
print(f'Shape: {df.shape}')
print('Columns:', df.columns.tolist())
df.head()

In [ ]:
# Normalise column names
df.columns = [c.strip() for c in df.columns]
name_col   = [c for c in df.columns if c.lower() in ('name', 'firstname', 'first_name')][0]
gender_col = [c for c in df.columns if c.lower() in ('gender', 'sex')][0]

print(f'Name column: "{name_col}"  |  Gender column: "{gender_col}"')
print('\nClass distribution:')
print(df[gender_col].value_counts())

## 2. Feature Engineering — Letter Frequency Matrix

In [ ]:
LETTERS = list('abcdefghijklmnopqrstuvwxyz')

def name_to_features(name: str) -> list:
    """Return a 26-dim vector of letter frequencies for a given name."""
    name = str(name).lower()
    return [name.count(ch) for ch in LETTERS]

# Build feature matrix
X = np.array([name_to_features(n) for n in df[name_col]])
y = df[gender_col].values

print(f'Feature matrix shape : {X.shape}')
print(f'Label array shape    : {y.shape}')
print(f'Classes              : {np.unique(y)}')

In [ ]:
# Mean letter frequency by gender (visualise)
feat_df = pd.DataFrame(X, columns=LETTERS)
feat_df['Gender'] = y
means = feat_df.groupby('Gender')[LETTERS].mean()

fig, ax = plt.subplots(figsize=(14, 4))
x_pos = np.arange(len(LETTERS))
width = 0.4
ax.bar(x_pos - width/2, means.loc['F'], width, label='F', color='#DD8452', alpha=0.85)
ax.bar(x_pos + width/2, means.loc['M'], width, label='M', color='#4C72B0', alpha=0.85)
ax.set_xticks(x_pos)
ax.set_xticklabels(LETTERS)
ax.set_title('Mean Letter Frequency per Name — by Gender')
ax.set_ylabel('Mean frequency')
ax.legend()
plt.tight_layout()
plt.show()

## 3. Train / Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_SEED, stratify=y
)
print(f'Train: {X_train.shape[0]:,}  |  Test: {X_test.shape[0]:,}')

## 4. Model Training & Comparison

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=RANDOM_SEED),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=7),
    'Naive Bayes':         MultinomialNB(),
    'Random Forest':       RandomForestClassifier(n_estimators=200, max_depth=20,
                                                   random_state=RANDOM_SEED, n_jobs=-1),
    'SVM (RBF)':           SVC(kernel='rbf', C=10, random_state=RANDOM_SEED),
}

results = {}
for name, clf in models.items():
    clf.fit(X_train, y_train)
    y_pred   = clf.predict(X_test)
    acc      = accuracy_score(y_test, y_pred)
    results[name] = {'model': clf, 'accuracy': acc, 'y_pred': y_pred}
    print(f'{name:<25}  accuracy = {acc:.4f}')

## 5. Cross-Validation on Best Model (Random Forest)

In [ ]:
best_clf = results['Random Forest']['model']

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=RANDOM_SEED)
cv_scores = cross_val_score(best_clf, X, y, cv=cv, scoring='accuracy', n_jobs=-1)

print(f'10-Fold CV Accuracy: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}')
print(f'Fold scores: {cv_scores.round(4)}')

## 6. Model Comparison Plot

In [ ]:
acc_values = {k: v['accuracy'] for k, v in results.items()}
sorted_acc = dict(sorted(acc_values.items(), key=lambda x: x[1], reverse=True))

fig, ax = plt.subplots(figsize=(10, 5))
colors = ['#2ecc71' if k == 'Random Forest' else '#3498db' for k in sorted_acc]
bars = ax.barh(list(sorted_acc.keys()), list(sorted_acc.values()), color=colors)
ax.set_xlim(0.5, 1.0)
ax.set_xlabel('Test Accuracy')
ax.set_title('Gender Prediction — Model Comparison')
for bar, val in zip(bars, sorted_acc.values()):
    ax.text(val + 0.002, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'docs', 'gender_model_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

## 7. Detailed Evaluation — Random Forest

In [ ]:
y_pred_rf = results['Random Forest']['y_pred']
print('Classification Report:')
print(classification_report(y_test, y_pred_rf, target_names=['F', 'M']))

In [ ]:
cm = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(cm, display_labels=['F', 'M'])
fig, ax = plt.subplots(figsize=(5, 4))
disp.plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Confusion Matrix — Gender (Random Forest)')
plt.tight_layout()
plt.savefig(os.path.join(BASE, 'docs', 'gender_confusion_matrix.png'), dpi=150, bbox_inches='tight')
plt.show()

## 8. Quick Sanity Check — Predict on Sample Names

In [ ]:
# Use the best in-memory model (Random Forest) to predict on sample names
best_clf = results['Random Forest']['model']

def predict_gender(first_name: str) -> str:
    """Return 'M' or 'F' for a given first name."""
    features = np.array([name_to_features(first_name)])
    return best_clf.predict(features)[0]

test_names = [
    ('Mary',      'F'),
    ('James',     'M'),
    ('Jennifer',  'F'),
    ('Michael',   'M'),
    ('Ashley',    'F'),
    ('Jordan',    'M'),   # ambiguous
    ('Taylor',    'F'),   # ambiguous
    ('Momin',     'M'),
]

print(f'{"Name":<15} {"Expected":<10} {"Predicted":<10} {"Correct"}')
print('-' * 45)
correct = 0
for name, expected in test_names:
    pred = predict_gender(name)
    ok   = pred == expected
    correct += ok
    print(f'{name:<15} {expected:<10} {pred:<10} {"✓" if ok else "✗"}')

print(f'\nTest accuracy: {correct}/{len(test_names)} ({correct/len(test_names)*100:.1f}%)')

## 9. Next Steps

This baseline achieves ~77–78% accuracy using simple letter counts.  
For the **production model** using character n-gram TF-IDF + LightGBM (~83–87% accuracy), see **notebook 05**.